# Stage 1 LS — NODDI-informed fitting strategies for the full Tort3 model

## Aim

Test the **full Tort3 model** under five increasingly strong uses of NODDI information, with `d_in`, `d_soma`, and `d_csf` fixed in every fit.

### Acquisition

`rm_full` directional acquisition.

### Truth scenarios

| Scenario | `f_soma` | `r_soma` | `f_fw` |
|---|---:|---:|---:|
| Truth 1 | 0.10 | 5 | 0.15 |
| Truth 2 | 0.15 | 7 | 0.20 |
| Truth 3 | 0.20 | 8 | 0.25 |

The true neurite:extra fraction ratio is kept at 2:1.

### Prescribed NODDI performance

Rather than repeatedly fitting NODDI, one NODDI estimate is generated **once per truth scenario** and then reused for every noisy repeat from that truth.

The requested mean absolute errors are based on conservatively rounded up estimate errors from NODDI runs on this simulated acquisition dataset:

- `kappa`: 0.10 (rounded up to 0.15)
- `f_extra`: 0.05 (rounded up to 0.06)
- `f_neurite`: 0.01 (rounded up to 0.02)
- `f_fw`: 0.03 (rounded up to 0.04)

A symmetric Laplace error model is used because MAE alone does not uniquely define a probability distribution. The seed is fixed so the same NODDI estimates are reused across the whole experiment.

### Fitting strategies

1. **Fully free**
2. **NODDI-initialised**
3. **NODDI-initialised + moderate regularisation**
4. **NODDI-initialised + tight regularisation**
5. **NODDI-fixed**

### Condition A

`r_soma`, `f_soma`, `f_neurite`, and `f_fw` remain genuinely free parameters.

NODDI informs:
- `kappa`
- `f_extra`

through initialisation and, for the regularised strategies, Gaussian-style penalty residuals.

`f_neurite` and `f_fw` are deliberately not regularised in A because they are being treated as quantities we may want to recover independently.

### Condition B

`r_soma` and `f_soma` are the biological parameters of interest.

The nuisance fractions and `kappa` are NODDI-informed:
- initialised from NODDI;
- regularised in the moderate/tight strategies;
- fixed/informatively constrained in the NODDI-fixed strategy.

The simplex is always enforced exactly.

### Important

The moderate/tight conditions are **LS analogues of Gaussian priors**: penalty residuals are appended to the signal residuals. They are not Bayesian posterior fits.

True Bayesian marginalisation is reserved for the later MCMC stage.


In [33]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dmri_models import (
    Ball,
    Component,
    Model,
    ModelPreset,
    Parameter,
    ParameterSpace,
    Simplex,
    Sphere,
    Stick,
)
from dmri_models.acquisitions import rm_full
from dmri_models.compartments import (
    WatsonFOD,
    FODConvolved,
    NODDISomaDispersionExtraNeurite,
)


In [ ]:
# ---------------------------------------------------------------------------
# Run settings
# ---------------------------------------------------------------------------

N_REPEATS = 1
BASE_SEED = 5000

N_STARTS = 1
MAX_NFEV = 30
SNR = 100.0

acquisition = rm_full()
acquisition.print_summary()

RESULTS_FILE = Path(
    "stage1_ls_noddi_informed_tort3_checkpoint.csv"
)


----------------------------------------------------------------
Acquisition summary
- measurements: 200
- b-value shells (ms/um^2): 7 (tolerance: 0.100)
  - b=0.000: 20 measurements, 0 non-zero directions
  - b=0.500: 30 measurements, 30 non-zero directions
  - b=1.000: 30 measurements, 30 non-zero directions
  - b=2.000: 30 measurements, 30 non-zero directions
  - b=3.000: 30 measurements, 30 non-zero directions
  - b=4.000: 30 measurements, 30 non-zero directions
  - b=5.000: 30 measurements, 30 non-zero directions
- gradient directions: 180 unique non-zero across 180 diffusion-weighted measurements
- b-tensor shape: 1.000
- pulse duration (ms): 5.000
- diffusion time (ms): 12.000
- gradient strength (T/m): 0.000 to 0.520
- echo time (ms): 26.000
- inversion time: n/a
- repetition time (ms): 2500.000
----------------------------------------------------------------


In [35]:
# ---------------------------------------------------------------------------
# Signal / noise helpers
# ---------------------------------------------------------------------------

def simulate_signal(preset, acquisition, parameters):
    return np.asarray(
        preset.model.signal(
            acquisition,
            parameters,
        ),
        dtype=float,
    )


def add_gaussian_noise(signal, snr, seed):
    signal = np.asarray(signal, dtype=float)
    rng = np.random.default_rng(seed)
    sigma = float(signal.max()) / float(snr)
    return signal + rng.normal(
        0.0,
        sigma,
        size=signal.shape,
    )


def odi_from_kappa(kappa: float) -> float:
    if not np.isfinite(kappa) or kappa < 0:
        raise ValueError(
            "kappa must be finite and non-negative."
        )
    return float(
        (2.0 / np.pi) * np.arctan2(1.0, kappa)
    )


In [36]:
# ---------------------------------------------------------------------------
# Truth data
# ---------------------------------------------------------------------------

TRUE_BASE = {
    "S0": 1.0,
    "d_in": 2.2,
    "d_perp": 0.5,
    "d_soma": 3.0,
    "d_csf": 3.0,
    "kappa": 2.0,
    "theta": 1.0,
    "phi": 1.2,
}

TRUTH_SCENARIOS = {
    "truth_1": {
        "label": "f_soma=0.10, r_soma=5, f_fw=0.15",
        "soma_fraction": 0.10,
        "r_soma": 5.0,
        "csf_fraction": 0.15,
    },
    "truth_2": {
        "label": "f_soma=0.15, r_soma=7, f_fw=0.20",
        "soma_fraction": 0.15,
        "r_soma": 7.0,
        "csf_fraction": 0.20,
    },
    "truth_3": {
        "label": "f_soma=0.20, r_soma=8, f_fw=0.25",
        "soma_fraction": 0.20,
        "r_soma": 8.0,
        "csf_fraction": 0.25,
    },
}


def make_truth(scenario, neurite_to_extra=2.0):
    f_soma = scenario["soma_fraction"]
    f_fw = scenario["csf_fraction"]

    remaining = 1.0 - f_soma - f_fw

    f_extra = remaining / (
        1.0 + neurite_to_extra
    )
    f_neurite = neurite_to_extra * f_extra

    truth = dict(TRUE_BASE)
    truth.update({
        "r_soma": scenario["r_soma"],
        "neurite_fraction": f_neurite,
        "extra_fraction": f_extra,
        "soma_fraction": f_soma,
        "csf_fraction": f_fw,
    })
    return truth


TRUTH_BY_SCENARIO = {
    name: make_truth(scenario)
    for name, scenario in TRUTH_SCENARIOS.items()
}

truth_df = pd.DataFrame([
    {
        "scenario": name,
        "r_soma": truth["r_soma"],
        "f_soma": truth["soma_fraction"],
        "f_fw": truth["csf_fraction"],
        "f_neurite": truth["neurite_fraction"],
        "f_extra": truth["extra_fraction"],
    }
    for name, truth in TRUTH_BY_SCENARIO.items()
])

display(truth_df)


,scenario,r_soma,f_soma,f_fw,f_neurite,f_extra
0,truth_1,5.0,0.10,0.15,0.500000,0.250000
1,truth_2,7.0,0.15,0.20,0.433333,0.216667
2,truth_3,8.0,0.20,0.25,0.366667,0.183333


In [37]:
# ---------------------------------------------------------------------------
# Prescribed NODDI performance
# ---------------------------------------------------------------------------

NODDI_MAE = {
    "kappa": 0.15,
    "extra_fraction": 0.06,
    "neurite_fraction": 0.02,
    "csf_fraction": 0.04,
}

NODDI_ESTIMATE_SEED = 24680


def make_one_noddi_estimate(truth, seed):
    rng = np.random.default_rng(seed)

    estimate = {
        "kappa": max(
            0.0,
            truth["kappa"]
            + rng.laplace(
                0.0,
                NODDI_MAE["kappa"],
            ),
        ),
        "neurite_fraction": max(
            0.0,
            truth["neurite_fraction"]
            + rng.laplace(
                0.0,
                NODDI_MAE["neurite_fraction"],
            ),
        ),
        "extra_fraction": max(
            0.0,
            truth["extra_fraction"]
            + rng.laplace(
                0.0,
                NODDI_MAE["extra_fraction"],
            ),
        ),
        "csf_fraction": max(
            0.0,
            truth["csf_fraction"]
            + rng.laplace(
                0.0,
                NODDI_MAE["csf_fraction"],
            ),
        ),
    }

    # Keep the NODDI-estimated nuisance fractions in a
    # physically valid non-soma simplex.
    nuisance = np.array([
        estimate["neurite_fraction"],
        estimate["extra_fraction"],
        estimate["csf_fraction"],
    ])

    if nuisance.sum() >= 0.98:
        nuisance *= 0.98 / nuisance.sum()

    estimate["neurite_fraction"] = float(nuisance[0])
    estimate["extra_fraction"] = float(nuisance[1])
    estimate["csf_fraction"] = float(nuisance[2])

    return estimate


NODDI_ESTIMATES = {
    name: make_one_noddi_estimate(
        truth,
        NODDI_ESTIMATE_SEED + i,
    )
    for i, (name, truth)
    in enumerate(TRUTH_BY_SCENARIO.items())
}


noddi_df = pd.DataFrame([
    {
        "scenario": name,
        **estimate,
        "odi": odi_from_kappa(
            estimate["kappa"]
        ),
    }
    for name, estimate
    in NODDI_ESTIMATES.items()
])

display(noddi_df)


,scenario,kappa,neurite_fraction,extra_fraction,csf_fraction,odi
0,truth_1,2.109114,0.497980,0.127816,0.176020,0.281857
1,truth_2,2.227177,0.449040,0.137876,0.139977,0.268667
2,truth_3,1.989899,0.377474,0.180658,0.302198,0.296459


In [38]:
# ---------------------------------------------------------------------------
# Full Tort3 model
# ---------------------------------------------------------------------------

def full_tort3_model(
    *,
    theta=1.0,
    phi=1.2,
    vary_r_soma=True,
    vary_kappa=True,
):
    fod = WatsonFOD(
        kappa="kappa",
        theta="theta",
        phi="phi",
        n_theta=20,
        n_phi=40,
    )

    model = Model([
        Component(
            "neurite",
            compartment=FODConvolved(
                response=Stick(
                    diffusivity="d_in",
                    orientation="axis",
                ),
                fod=fod,
                orientation_parameter="axis",
            ),
        ),
        Component(
            "extra",
            compartment=FODConvolved(
                response=NODDISomaDispersionExtraNeurite(
                    parallel_diffusivity="d_in",
                    orientation="axis",
                    neurite_fraction="neurite_fraction",
                    soma_fraction="soma_fraction",
                    csf_fraction="csf_fraction",
                    kappa="kappa",
                ),
                fod=fod,
                orientation_parameter="axis",
            ),
        ),
        Component(
            "soma",
            compartment=Sphere(
                radius="r_soma",
                diffusivity="d_soma",
            ),
        ),
        Component(
            "csf",
            compartment=Ball(
                diffusivity="d_csf",
            ),
        ),
    ])

    parameters = ParameterSpace(
        [
            Parameter("S0", 1.0, (0.05, 2.0)),
            Parameter(
                "d_in",
                2.2,
                (0.5, 3.0),
                vary=False,
            ),
            Parameter(
                "d_soma",
                3.0,
                (0.5, 3.0),
                vary=False,
            ),
            Parameter(
                "d_csf",
                3.0,
                (0.5, 3.0),
                vary=False,
            ),
            Parameter(
                "r_soma",
                7.0,
                (1.0, 15.0),
                vary=vary_r_soma,
            ),
            Parameter(
                "kappa",
                2.0,
                (0.0, 64.0),
                vary=vary_kappa,
            ),
            Parameter(
                "theta",
                theta,
                (0.0, np.pi),
                vary=False,
            ),
            Parameter(
                "phi",
                phi,
                (-np.pi, np.pi),
                vary=False,
            ),
        ],
        simplexes=[
            Simplex(
                (
                    "neurite_fraction",
                    "extra_fraction",
                    "soma_fraction",
                    "csf_fraction",
                ),
                (
                    0.50,
                    0.25,
                    0.10,
                    0.15,
                ),
            )
        ],
    )

    return ModelPreset(
        model=model,
        parameters=parameters,
    )


## Parameterisation used by the Stage 1 fitter

The fitted parameters are:

- `S0` — always free nuisance scale.
- `r_soma` — always free.
- `soma_fraction` — always free.
- `kappa` — free for strategies 1–4; NODDI-fixed for strategy 5.
- Fractions — always obey the exact simplex constraint.

For the non-fixed strategies, the four fractions are represented with four softmax logits. This guarantees:

`f_neurite + f_extra + f_soma + f_fw = 1`

at every optimisation point.

For **Condition A**, all three non-soma fractions remain free; NODDI regularisation is applied only to nuisance `f_extra` and `kappa`.

For **Condition B**, the non-soma fractions and `kappa` are all NODDI-informed; the moderate/tight strategies penalise deviations of `f_neurite`, `f_extra`, `f_fw`, and `kappa`.

For **NODDI-fixed**, the non-soma composition and `kappa` are fixed/informatively constrained to the NODDI estimate, while `f_soma` remains free. The non-soma NODDI fractions are rescaled to sum to `1 - f_soma`, preserving the simplex.


In [39]:
STRATEGIES = [
    ("A", "free"),
    ("A", "noddi_initialised"),
    ("A", "moderate"),
    ("A", "tight"),
    ("A", "fixed"),
    ("B", "free"),
    ("B", "noddi_initialised"),
    ("B", "moderate"),
    ("B", "tight"),
    ("B", "fixed"),
]

STRATEGY_LABELS = {
    "free": "Fully free",
    "noddi_initialised": "NODDI initialised",
    "moderate": "NODDI + moderate prior",
    "tight": "NODDI + tight prior",
    "fixed": "NODDI fixed",
}


def regularisation_scales(strategy):
    if strategy == "moderate":
        multiplier = 2.0
    elif strategy == "tight":
        multiplier = 0.5
    else:
        return {}

    return {
        key: multiplier * value
        for key, value in NODDI_MAE.items()
    }


def logits_to_simplex(logits):
    logits = np.asarray(logits, dtype=float)
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum()


def simplex_to_logits(fractions):
    fractions = np.asarray(
        fractions,
        dtype=float,
    )
    fractions = np.clip(
        fractions,
        1e-8,
        None,
    )
    fractions = fractions / fractions.sum()
    return np.log(fractions)


In [40]:
def fit_stage1(
    data,
    truth,
    noddi,
    *,
    condition,
    strategy,
    seed=0,
):
    rng = np.random.default_rng(seed)

    # ------------------------------------------------------------------
    # Starting values
    # ------------------------------------------------------------------

    if strategy == "free":
        # Do NOT initialise from truth.
        # Use neutral/default values instead.
        init = {
            "S0": 1.0,
            "r_soma": 7.0,
            "kappa": 2.0,
            "neurite_fraction": 0.50,
            "extra_fraction": 0.25,
            "soma_fraction": 0.10,
            "csf_fraction": 0.15,
        }

    else:
        init = {
            "S0": 1.0,
            "r_soma": 7.0,
            "kappa": noddi["kappa"],
            "neurite_fraction": noddi["neurite_fraction"],
            "extra_fraction": noddi["extra_fraction"],
            "soma_fraction": 0.10,
            "csf_fraction": noddi["csf_fraction"],
        }

        # Keep the NODDI-informed non-soma composition while making the
        # starting soma fraction a neutral 0.10.
        nuisance = np.array([
            init["neurite_fraction"],
            init["extra_fraction"],
            init["csf_fraction"],
        ])
        nuisance *= (
            (1.0 - init["soma_fraction"])
            / nuisance.sum()
        )

        init["neurite_fraction"] = float(nuisance[0])
        init["extra_fraction"] = float(nuisance[1])
        init["csf_fraction"] = float(nuisance[2])

    # NODDI-fixed uses the same starting values but removes nuisance
    # freedom below.
    kappa_free = strategy != "fixed"

    # ------------------------------------------------------------------
    # Parameter vector
    #
    # Always:
    #   S0, r_soma, [kappa], 4 fraction logits
    #
    # For strategy="fixed", kappa and the non-soma composition are
    # replaced by the NODDI estimate inside unpack().
    # ------------------------------------------------------------------

    x0 = [
        init["S0"],
        init["r_soma"],
    ]

    if kappa_free:
        x0.append(init["kappa"])

    x0.extend(
        simplex_to_logits([
            init["neurite_fraction"],
            init["extra_fraction"],
            init["soma_fraction"],
            init["csf_fraction"],
        ])
    )

    lower = [
        0.05,
        1.0,
    ]
    upper = [
        2.0,
        15.0,
    ]

    if kappa_free:
        lower.append(0.0)
        upper.append(64.0)

    lower.extend([-20.0] * 4)
    upper.extend([20.0] * 4)

    preset = full_tort3_model(
        vary_r_soma=True,
        vary_kappa=True,
    )

    sigma_data = float(
        np.max(np.abs(data)) / SNR
    )

    prior_scales = regularisation_scales(
        strategy
    )

    # ------------------------------------------------------------------
    # Convert optimiser vector -> physical parameters
    # ------------------------------------------------------------------

    def unpack(x):
        idx = 0

        values = dict(TRUE_BASE)
        values["S0"] = float(x[idx])
        idx += 1

        values["r_soma"] = float(x[idx])
        idx += 1

        if kappa_free:
            values["kappa"] = float(x[idx])
            idx += 1
        else:
            values["kappa"] = float(
                noddi["kappa"]
            )

        logits = x[idx:idx + 4]
        fractions = logits_to_simplex(logits)

        f_neurite = float(fractions[0])
        f_extra = float(fractions[1])
        f_soma = float(fractions[2])
        f_fw = float(fractions[3])

        if strategy == "fixed":
            # NODDI fixes the non-soma composition, but because f_soma
            # remains a target, the NODDI nuisance composition is rescaled
            # to fill exactly 1 - f_soma.
            nuisance = np.array([
                noddi["neurite_fraction"],
                noddi["extra_fraction"],
                noddi["csf_fraction"],
            ])

            available = max(
                1e-8,
                1.0 - f_soma,
            )

            nuisance *= (
                available / nuisance.sum()
            )

            f_neurite = float(nuisance[0])
            f_extra = float(nuisance[1])
            f_fw = float(nuisance[2])

        values["neurite_fraction"] = f_neurite
        values["extra_fraction"] = f_extra
        values["soma_fraction"] = f_soma
        values["csf_fraction"] = f_fw

        # These are fixed in every Stage 1 fit.
        values["d_in"] = truth["d_in"]
        values["d_soma"] = truth["d_soma"]
        values["d_csf"] = truth["d_csf"]

        return values

    # ------------------------------------------------------------------
    # Residual function
    # ------------------------------------------------------------------

    def residual(x):
        values = unpack(x)

        prediction = simulate_signal(
            preset,
            acquisition,
            values,
        )

        residuals = [
            *(
                (prediction - data)
                / sigma_data
            )
        ]

        # --------------------------------------------------------------
        # Gaussian-style prior penalties
        # --------------------------------------------------------------

        if strategy in {
            "moderate",
            "tight",
        }:

            prior_parameters = [
                "kappa",
                "extra_fraction",
            ]

            # Condition B treats all nuisance fractions as low-importance.
            if condition == "B":
                prior_parameters += [
                    "neurite_fraction",
                    "csf_fraction",
                ]

            for parameter in prior_parameters:
                residuals.append(
                    (
                        values[parameter]
                        - noddi[parameter]
                    )
                    / prior_scales[parameter]
                )

        return np.asarray(
            residuals,
            dtype=float,
        )

    # ------------------------------------------------------------------
    # Multi-start least squares
    # ------------------------------------------------------------------

    starts = [
        np.asarray(
            x0,
            dtype=float,
        )
    ]

    for _ in range(
        max(0, N_STARTS - 1)
    ):

        start = np.asarray(
            x0,
            dtype=float,
        ).copy()

        # Perturb S0 and r_soma.
        start[0] += rng.normal(
            0.0,
            0.03,
        )
        start[1] += rng.normal(
            0.0,
            1.0,
        )

        # Perturb kappa when free.
        if kappa_free:
            start[2] = max(
                0.0,
                start[2]
                + rng.normal(0.0, 1.0),
            )
            logit_start = 3
        else:
            logit_start = 2

        # Perturb fraction logits too.
        start[logit_start:] += rng.normal(
            0.0,
            0.25,
            size=4,
        )

        start = np.clip(
            start,
            np.asarray(lower),
            np.asarray(upper),
        )

        starts.append(start)

    candidates = []

    for start in starts:
        try:
            result = least_squares(
                residual,
                start,
                bounds=(
                    lower,
                    upper,
                ),
                max_nfev=MAX_NFEV,
            )
            candidates.append(result)
        except ValueError:
            # A pathological optimisation start should not kill the whole run.
            continue

    if not candidates:
        raise RuntimeError(
            "All optimisation starts failed."
        )

    best = min(
        candidates,
        key=lambda result: result.cost,
    )

    fitted = unpack(best.x)

    prediction = simulate_signal(
        preset,
        acquisition,
        fitted,
    )

    return {
        **fitted,
        "success": bool(
            best.success
        ),
        "n_evaluations": int(
            best.nfev
        ),
        "rmse": float(
            np.sqrt(
                np.mean(
                    (
                        prediction
                        - data
                    ) ** 2
                )
            )
        ),
    }


## Main paired benchmark

The same noisy dataset is passed to every strategy within a truth/repeat.

The NODDI estimate is also fixed for each truth scenario and reused across all repeats for that truth, as specified.

Results are checkpointed after every fit so an overnight interruption does not discard completed fits.


In [41]:
results = []

for scenario_name, truth in TRUTH_BY_SCENARIO.items():

    print(
        f"\nScenario: {scenario_name} - "
        f"{TRUTH_SCENARIOS[scenario_name]['label']}",
        flush=True,
    )

    truth_preset = full_tort3_model(
        vary_r_soma=False,
        vary_kappa=False,
    )

    truth_signal = simulate_signal(
        truth_preset,
        acquisition,
        truth,
    )

    noddi = NODDI_ESTIMATES[
        scenario_name
    ]

    for rep in range(
        N_REPEATS
    ):

        seed = BASE_SEED + rep

        data_noisy = add_gaussian_noise(
            truth_signal,
            SNR,
            seed,
        )

        print(
            f"  repeat {rep + 1}/{N_REPEATS}",
            flush=True,
        )

        for condition, strategy in STRATEGIES:

            print(
                f"    {condition} / "
                f"{STRATEGY_LABELS[strategy]}",
                flush=True,
            )

            t0 = time.time()

            try:
                fit = fit_stage1(
                    data_noisy,
                    truth,
                    noddi,
                    condition=condition,
                    strategy=strategy,
                    seed=seed,
                )

                row = {
                    "scenario": scenario_name,
                    "condition": condition,
                    "strategy": strategy,
                    "strategy_label": STRATEGY_LABELS[strategy],
                    "repeat": rep,
                    "seed": seed,
                    "success": fit["success"],
                    "rmse": fit["rmse"],
                    "n_evaluations": fit["n_evaluations"],
                    "runtime_s": (
                        time.time() - t0
                    ),
                    "true_r_soma": truth["r_soma"],
                    "true_soma_fraction": truth["soma_fraction"],
                    "true_f_fw": truth["csf_fraction"],
                    "noddi_kappa": noddi["kappa"],
                    "noddi_neurite_fraction": noddi["neurite_fraction"],
                    "noddi_extra_fraction": noddi["extra_fraction"],
                    "noddi_csf_fraction": noddi["csf_fraction"],
                }

                for parameter in [
                    "S0",
                    "r_soma",
                    "kappa",
                    "neurite_fraction",
                    "extra_fraction",
                    "soma_fraction",
                    "csf_fraction",
                ]:
                    if parameter in fit:
                        row[parameter] = fit[parameter]

                row["fraction_sum"] = (
                    row["neurite_fraction"]
                    + row["extra_fraction"]
                    + row["soma_fraction"]
                    + row["csf_fraction"]
                )

                row["r_soma_at_lower_bound"] = np.isclose(
                    row["r_soma"],
                    1.0,
                )

                row["r_soma_at_upper_bound"] = np.isclose(
                    row["r_soma"],
                    15.0,
                )

            except Exception as exc:

                row = {
                    "scenario": scenario_name,
                    "condition": condition,
                    "strategy": strategy,
                    "strategy_label": STRATEGY_LABELS[strategy],
                    "repeat": rep,
                    "seed": seed,
                    "success": False,
                    "rmse": np.nan,
                    "n_evaluations": np.nan,
                    "runtime_s": (
                        time.time() - t0
                    ),
                    "error": repr(exc),
                    "true_r_soma": truth["r_soma"],
                    "true_soma_fraction": truth["soma_fraction"],
                    "true_f_fw": truth["csf_fraction"],
                    "noddi_kappa": noddi["kappa"],
                    "noddi_neurite_fraction": noddi["neurite_fraction"],
                    "noddi_extra_fraction": noddi["extra_fraction"],
                    "noddi_csf_fraction": noddi["csf_fraction"],
                }

                print(
                    f"      FAILED: {exc}",
                    flush=True,
                )

            results.append(row)

            pd.DataFrame(
                results
            ).to_csv(
                RESULTS_FILE,
                index=False,
            )

results_df = pd.DataFrame(
    results
)

print(
    "Rows:",
    len(results_df),
)

display(
    results_df.head()
)



Scenario: truth_1 - f_soma=0.10, r_soma=5, f_fw=0.15
  repeat 1/20
    A / Fully free
    A / NODDI initialised
    A / NODDI + moderate prior
    A / NODDI + tight prior
    A / NODDI fixed
    B / Fully free
    B / NODDI initialised
    B / NODDI + moderate prior
    B / NODDI + tight prior
    B / NODDI fixed
  repeat 2/20
    A / Fully free
    A / NODDI initialised
    A / NODDI + moderate prior
    A / NODDI + tight prior
    A / NODDI fixed
    B / Fully free
    B / NODDI initialised
    B / NODDI + moderate prior
    B / NODDI + tight prior
    B / NODDI fixed
  repeat 3/20
    A / Fully free
    A / NODDI initialised
    A / NODDI + moderate prior
    A / NODDI + tight prior
    A / NODDI fixed
    B / Fully free
    B / NODDI initialised
    B / NODDI + moderate prior
    B / NODDI + tight prior
    B / NODDI fixed
  repeat 4/20
    A / Fully free
    A / NODDI initialised
    A / NODDI + moderate prior
    A / NODDI + tight prior
    A / NODDI fixed
    B / Fully free
   

,scenario,condition,strategy,strategy_label,repeat,seed,success,rmse,n_evaluations,runtime_s,...,S0,r_soma,kappa,neurite_fraction,extra_fraction,soma_fraction,csf_fraction,fraction_sum,r_soma_at_lower_bound,r_soma_at_upper_bound
0,truth_1,A,free,Fully free,0,5000,True,0.010062,17,32.057709,...,0.997584,7.622184,1.728495,0.603603,0.101153,0.117192,0.178052,1.0,False,False
1,truth_1,A,noddi_initialised,NODDI initialised,0,5000,True,0.010062,33,29.146103,...,0.997584,7.622184,1.728495,0.603603,0.101153,0.117192,0.178052,1.0,False,False
2,truth_1,A,moderate,NODDI + moderate prior,0,5000,True,0.010062,27,36.849310,...,0.997599,7.548310,1.734605,0.602167,0.103720,0.115418,0.178695,1.0,False,False
3,truth_1,A,tight,NODDI + tight prior,0,5000,True,0.010229,20,26.221251,...,0.998119,5.423190,2.011808,0.510892,0.193196,0.114015,0.181897,1.0,False,False
4,truth_1,A,fixed,NODDI fixed,0,5000,True,0.011311,22,916.285036,...,0.993948,6.145492,2.109114,0.535754,0.137511,0.137363,0.189372,1.0,False,False


## Recovery metrics

In [42]:
PRIMARY_PARAMS = {
    "r_soma": "true_r_soma",
    "soma_fraction": "true_soma_fraction",
    "csf_fraction": "true_f_fw",
}

metric_rows = []

for (
    condition,
    strategy,
    scenario,
), sub in results_df.groupby(
    ["condition", "strategy", "scenario"]
):

    for parameter, truth_column in PRIMARY_PARAMS.items():

        if parameter not in sub.columns:
            continue

        values = pd.to_numeric(
            sub[parameter],
            errors="coerce",
        )

        truth_value = float(
            sub[truth_column].iloc[0]
        )

        valid = values.notna()

        if valid.sum() == 0:
            continue

        errors = (
            values[valid]
            - truth_value
        )

        metric_rows.append({
            "condition": condition,
            "strategy": strategy,
            "strategy_label": STRATEGY_LABELS[strategy],
            "scenario": scenario,
            "parameter": parameter,
            "truth": truth_value,
            "mean": values[valid].mean(),
            "bias": errors.mean(),
            "mae": errors.abs().mean(),
            "rmse": np.sqrt(
                np.mean(errors ** 2)
            ),
            "sd": values[valid].std(
                ddof=1
            ),
            "relative_rmse": (
                np.sqrt(
                    np.mean(
                        errors ** 2
                    )
                )
                / abs(truth_value)
                if truth_value != 0
                else np.nan
            ),
            "n_valid": int(
                valid.sum()
            ),
            "n_total": len(values),
        })

metrics_df = pd.DataFrame(
    metric_rows
)

display(
    metrics_df.sort_values(
        [
            "condition",
            "scenario",
            "parameter",
            "rmse",
        ]
    )
)


,condition,strategy,strategy_label,scenario,parameter,truth,mean,bias,mae,rmse,sd,relative_rmse,n_valid,n_total
20,A,moderate,NODDI + moderate prior,truth_1,csf_fraction,0.15,0.152368,0.002368,0.009875,0.013013,0.013128,0.086751,20,20
29,A,noddi_initialised,NODDI initialised,truth_1,csf_fraction,0.15,0.150514,0.000514,0.010685,0.014309,0.014671,0.095392,20,20
11,A,free,Fully free,truth_1,csf_fraction,0.15,0.150514,0.000514,0.010686,0.014309,0.014671,0.095395,20,20
38,A,tight,NODDI + tight prior,truth_1,csf_fraction,0.15,0.172870,0.022870,0.022870,0.024369,0.008633,0.162459,20,20
2,A,fixed,NODDI fixed,truth_1,csf_fraction,0.15,0.189030,0.039030,0.039030,0.039059,0.001561,0.260395,20,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52,B,fixed,NODDI fixed,truth_3,soma_fraction,0.20,0.205657,0.005657,0.006578,0.008791,0.006903,0.043953,20,20
88,B,tight,NODDI + tight prior,truth_3,soma_fraction,0.20,0.212559,0.012559,0.012559,0.014333,0.007087,0.071666,20,20
70,B,moderate,NODDI + moderate prior,truth_3,soma_fraction,0.20,0.221997,0.021997,0.034180,0.040998,0.035496,0.204990,20,20
61,B,free,Fully free,truth_3,soma_fraction,0.20,0.239899,0.039899,0.063184,0.078457,0.069310,0.392287,20,20


In [43]:
summary = (
    metrics_df
    .groupby(
        [
            "condition",
            "strategy",
            "strategy_label",
            "parameter",
        ]
    )["relative_rmse"]
    .mean()
    .reset_index()
)

summary_pivot = (
    summary
    .pivot_table(
        index=[
            "condition",
            "strategy",
            "strategy_label",
        ],
        columns="parameter",
        values="relative_rmse",
    )
)

for parameter in [
    "r_soma",
    "soma_fraction",
    "csf_fraction",
]:
    if parameter not in summary_pivot.columns:
        summary_pivot[parameter] = np.nan

summary_pivot[
    "mean_soma_relative_rmse"
] = (
    summary_pivot[
        [
            "r_soma",
            "soma_fraction",
        ]
    ].mean(axis=1)
)

summary_pivot[
    "mean_primary_relative_rmse"
] = (
    summary_pivot[
        [
            "r_soma",
            "soma_fraction",
            "csf_fraction",
        ]
    ].mean(axis=1)
)

display(
    summary_pivot.sort_values(
        [
            "condition",
            "mean_soma_relative_rmse",
        ]
    )
)


parameter                                           csf_fraction    r_soma  \
condition strategy          strategy_label                                   
A         tight             NODDI + tight prior         0.105256  0.056972   
          moderate          NODDI + moderate prior      0.074503  0.128202   
          noddi_initialised NODDI initialised           0.087466  0.144626   
          free              Fully free                  0.087470  0.144640   
          fixed             NODDI fixed                 0.219331  0.206152   
B         tight             NODDI + tight prior         0.076518  0.064656   
          moderate          NODDI + moderate prior      0.065383  0.059767   
          noddi_initialised NODDI initialised           0.087466  0.144626   
          free              Fully free                  0.087470  0.144640   
          fixed             NODDI fixed                 0.219331  0.206152   

parameter                                           soma_fraction  \
condition strategy          strategy_label                          
A         tight             NODDI + tight prior          0.178499   
          moderate          NODDI + moderate prior       0.224717   
          noddi_initialised NODDI initialised            0.338884   
          free              Fully free                   0.338882   
          fixed             NODDI fixed                  0.376428   
B         tight             NODDI + tight prior          0.174840   
          moderate          NODDI + moderate prior       0.195683   
          noddi_initialised NODDI initialised            0.338884   
          free              Fully free                   0.338882   
          fixed             NODDI fixed                  0.376428   

parameter                                           mean_soma_relative_rmse  \
condition strategy          strategy_label                                    
A         tight             NODDI + tight prior                    0.117735   
          moderate          NODDI + moderate prior                 0.176460   
          noddi_initialised NODDI initialised                      0.241755   
          free              Fully free                             0.241761   
          fixed             NODDI fixed                            0.291290   
B         tight             NODDI + tight prior                    0.119748   
          moderate          NODDI + moderate prior                 0.127725   
          noddi_initialised NODDI initialised                      0.241755   
          free              Fully free                             0.241761   
          fixed             NODDI fixed                            0.291290   

parameter                                           mean_primary_relative_rmse  
condition strategy          strategy_label                                      
A         tight             NODDI + tight prior                       0.113576  
          moderate          NODDI + moderate prior                    0.142474  
          noddi_initialised NODDI initialised                         0.190325  
          free              Fully free                                0.190331  
          fixed             NODDI fixed                               0.267304  
B         tight             NODDI + tight prior                       0.105338  
          moderate          NODDI + moderate prior                    0.106944  
          noddi_initialised NODDI initialised                         0.190325  
          free              Fully free                                0.190331  
          fixed             NODDI fixed                               0.267304

In [46]:
# ============================================================================
# Summary table:
#   - absolute RMSE for each parameter
#   - mean relative RMSE across primary parameters
# ============================================================================

# Absolute RMSE for each parameter
absolute_summary = (
    metrics_df
    .groupby(
        [
            "condition",
            "strategy",
            "strategy_label",
            "parameter",
        ]
    )["rmse"]
    .mean()
    .reset_index()
)

absolute_pivot = (
    absolute_summary
    .pivot_table(
        index=[
            "condition",
            "strategy",
            "strategy_label",
        ],
        columns="parameter",
        values="rmse",
    )
)

# Relative RMSE for calculating the combined score
relative_summary = (
    metrics_df
    .groupby(
        [
            "condition",
            "strategy",
            "strategy_label",
            "parameter",
        ]
    )["relative_rmse"]
    .mean()
    .reset_index()
)

relative_pivot = (
    relative_summary
    .pivot_table(
        index=[
            "condition",
            "strategy",
            "strategy_label",
        ],
        columns="parameter",
        values="relative_rmse",
    )
)

PARAMS = [
    "r_soma",
    "soma_fraction",
    "csf_fraction",
]

# Make sure all expected parameters exist
for parameter in PARAMS:
    if parameter not in absolute_pivot.columns:
        absolute_pivot[parameter] = np.nan

    if parameter not in relative_pivot.columns:
        relative_pivot[parameter] = np.nan


# Mean PRIMARY RELATIVE RMSE
absolute_pivot["mean_primary_relative_rmse"] = (
    relative_pivot[PARAMS].mean(axis=1)
)


# Rename parameter columns for clarity
summary_pivot = absolute_pivot.rename(
    columns={
        "r_soma": "r_soma_RMSE",
        "soma_fraction": "f_soma_RMSE",
        "csf_fraction": "f_FW_RMSE",
    }
)


# Sort by overall relative performance within condition
summary_pivot = summary_pivot.sort_values(
    [
        "condition",
        "mean_primary_relative_rmse",
    ]
)

display(summary_pivot)

parameter                                           f_FW_RMSE  r_soma_RMSE  \
condition strategy          strategy_label                                   
A         tight             NODDI + tight prior      0.018957     0.370438   
          moderate          NODDI + moderate prior   0.014408     0.773124   
          noddi_initialised NODDI initialised        0.017154     0.884738   
          free              Fully free               0.017155     0.884826   
          fixed             NODDI fixed              0.041467     1.311670   
B         tight             NODDI + tight prior      0.013156     0.422539   
          moderate          NODDI + moderate prior   0.012677     0.415968   
          noddi_initialised NODDI initialised        0.017154     0.884738   
          free              Fully free               0.017155     0.884826   
          fixed             NODDI fixed              0.041467     1.311670   

parameter                                           f_soma_RMSE  \
condition strategy          strategy_label                        
A         tight             NODDI + tight prior        0.024786   
          moderate          NODDI + moderate prior     0.034255   
          noddi_initialised NODDI initialised          0.052409   
          free              Fully free                 0.052409   
          fixed             NODDI fixed                0.050608   
B         tight             NODDI + tight prior        0.024665   
          moderate          NODDI + moderate prior     0.031097   
          noddi_initialised NODDI initialised          0.052409   
          free              Fully free                 0.052409   
          fixed             NODDI fixed                0.050608   

parameter                                           mean_primary_relative_rmse  
condition strategy          strategy_label                                      
A         tight             NODDI + tight prior                       0.113576  
          moderate          NODDI + moderate prior                    0.142474  
          noddi_initialised NODDI initialised                         0.190325  
          free              Fully free                                0.190331  
          fixed             NODDI fixed                               0.267304  
B         tight             NODDI + tight prior                       0.105338  
          moderate          NODDI + moderate prior                    0.106944  
          noddi_initialised NODDI initialised                         0.190325  
          free              Fully free                                0.190331  
          fixed             NODDI fixed                               0.267304

In [44]:
robustness = (
    results_df
    .groupby(
        [
            "condition",
            "strategy",
            "strategy_label",
        ]
    )
    .agg(
        success_rate=(
            "success",
            "mean",
        ),
        signal_rmse_mean=(
            "rmse",
            "mean",
        ),
        signal_rmse_sd=(
            "rmse",
            "std",
        ),
        runtime_median_s=(
            "runtime_s",
            "median",
        ),
        r_soma_lower_bound_rate=(
            "r_soma_at_lower_bound",
            "mean",
        ),
        r_soma_upper_bound_rate=(
            "r_soma_at_upper_bound",
            "mean",
        ),
    )
    .reset_index()
)

display(
    robustness.sort_values(
        [
            "condition",
            "signal_rmse_mean",
        ]
    )
)


,condition,strategy,strategy_label,success_rate,signal_rmse_mean,signal_rmse_sd,runtime_median_s,r_soma_lower_bound_rate,r_soma_upper_bound_rate
3,A,noddi_initialised,NODDI initialised,1.0,0.009963,0.000591,30.367037,0.0,0.0
1,A,free,Fully free,1.0,0.009963,0.000591,30.747104,0.0,0.0
2,A,moderate,NODDI + moderate prior,1.0,0.009967,0.000590,29.447695,0.0,0.0
4,A,tight,NODDI + tight prior,1.0,0.010050,0.000588,23.892391,0.0,0.0
0,A,fixed,NODDI fixed,1.0,0.011481,0.001071,13.974424,0.0,0.0
8,B,noddi_initialised,NODDI initialised,1.0,0.009963,0.000591,32.539405,0.0,0.0
6,B,free,Fully free,1.0,0.009963,0.000591,30.846541,0.0,0.0
7,B,moderate,NODDI + moderate prior,1.0,0.009973,0.000587,26.554150,0.0,0.0
9,B,tight,NODDI + tight prior,1.0,0.010062,0.000582,22.703986,0.0,0.0
5,B,fixed,NODDI fixed,1.0,0.011481,0.001071,14.072377,0.0,0.0


In [45]:
# ---------------------------------------------------------------------------
# Recovery plots
# ---------------------------------------------------------------------------

PLOT_DIR = Path(
    "stage1_ls_noddi_informed_tort3_plots"
)
PLOT_DIR.mkdir(
    exist_ok=True
)

PARAM_LABELS = {
    "r_soma": "r_soma",
    "soma_fraction": "f_soma",
    "csf_fraction": "f_FW",
}

for condition in ["A", "B"]:

    for parameter in [
        "r_soma",
        "soma_fraction",
        "csf_fraction",
    ]:

        plot_data = metrics_df[
            (metrics_df["condition"] == condition)
            & (
                metrics_df["parameter"]
                == parameter
            )
        ].copy()

        if plot_data.empty:
            continue

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(18, 5),
            constrained_layout=True,
        )

        for ax, (
            scenario_name,
            scenario,
        ) in zip(
            axes,
            TRUTH_SCENARIOS.items(),
        ):

            sub = plot_data[
                plot_data["scenario"]
                == scenario_name
            ].copy()

            if sub.empty:
                ax.axis("off")
                continue

            x = np.arange(
                len(sub)
            )

            ax.errorbar(
                x,
                sub["mean"],
                yerr=sub["sd"],
                fmt="o",
                capsize=4,
            )

            ax.axhline(
                sub["truth"].iloc[0],
                linestyle="--",
                linewidth=1.5,
            )

            ax.set_xticks(x)
            ax.set_xticklabels(
                sub["strategy_label"],
                rotation=45,
                ha="right",
            )
            ax.set_ylabel(
                PARAM_LABELS[parameter]
            )
            ax.set_title(
                TRUTH_SCENARIOS[
                    scenario_name
                ]["label"]
            )
            ax.grid(
                True,
                alpha=0.25,
            )

        fig.suptitle(
            f"Condition {condition}: "
            f"recovery of "
            f"{PARAM_LABELS[parameter]}",
            y=1.02,
        )

        filename = (
            PLOT_DIR
            / (
                f"condition_{condition}_"
                f"{parameter}.png"
            )
        )

        fig.savefig(
            filename,
            dpi=200,
            bbox_inches="tight",
        )

        plt.close(fig)

        print(
            f"Saved: {filename}"
        )


Saved: stage1_ls_noddi_informed_tort3_plots/condition_A_r_soma.png
Saved: stage1_ls_noddi_informed_tort3_plots/condition_A_soma_fraction.png
Saved: stage1_ls_noddi_informed_tort3_plots/condition_A_csf_fraction.png
Saved: stage1_ls_noddi_informed_tort3_plots/condition_B_r_soma.png
Saved: stage1_ls_noddi_informed_tort3_plots/condition_B_soma_fraction.png
Saved: stage1_ls_noddi_informed_tort3_plots/condition_B_csf_fraction.png


## What to look for

The main comparison is whether NODDI information improves recovery of `r_soma` and `f_soma` without simply forcing the answer toward the NODDI estimates.

Important diagnostics:

- **Bias:** systematic displacement from the known truth.
- **RMSE:** combined accuracy and variance.
- **SD across repeats:** sensitivity to noise.
- **Signal RMSE:** whether a parameter improvement comes at the cost of worse signal fit.
- **Boundary-hit rate:** especially `r_soma=1` or `15`.
- **Condition A vs B:** whether keeping `f_neurite` and `f_fw` genuinely free materially destabilises soma recovery.
- **Moderate vs tight priors:** whether soft NODDI information is sufficient, or whether very strong constraints are needed.

The later MCMC experiment should use the most informative finalists to determine whether apparent improvement comes from genuinely better identifiability or simply from imposing stronger prior information.
